In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter
import time
from datetime import datetime

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используется устройство: {device}")

Используется устройство: cuda


In [9]:
# Определение преобразований для набора данных CIFAR-10
transform_cifar = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Загрузка набора данных
cifar10_train = datasets.CIFAR10(root='data', train=True, 
                                  transform=transform_cifar, download=True)
cifar10_loader_train = DataLoader(cifar10_train, batch_size=1024, shuffle=True)

# Параметры модели
input_size = 32 * 32 * 3  # 3072
output_size = 10
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Модель
model = nn.Linear(input_size, output_size).to(device)

# Функция потерь для многоклассовой классификации
loss_fn = nn.CrossEntropyLoss()  # Вместо BCELoss

optimizer = optim.Adam(model.parameters(), lr=0.001)

# Обучение
for epoch in range(10):
    running_loss = 0.0
    for x_batch, y_batch in cifar10_loader_train:
        # Перемещаем данные на устройство
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)
        
        # Изменяем форму: (batch_size, 3, 32, 32) -> (batch_size, 3072)
        x_batch = x_batch.view(x_batch.size(0), -1)
        
        # Прямой проход
        pred = model(x_batch)  # Форма: (batch_size, 10)
        
        # Вычисляем loss (CrossEntropyLoss ожидает классы, не one-hot)
        loss = loss_fn(pred, y_batch)
        
        # Обратный проход
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        
        running_loss += loss.item()
    
    avg_loss = running_loss / len(cifar10_loader_train)
    print(f"epoch = {epoch}, loss = {avg_loss:.4f}")

Files already downloaded and verified
epoch = 0, loss = 1.9944
epoch = 1, loss = 1.9081
epoch = 2, loss = 1.8950
epoch = 3, loss = 1.8895
epoch = 4, loss = 1.8887
epoch = 5, loss = 1.8806
epoch = 6, loss = 1.8812
epoch = 7, loss = 1.8828
epoch = 8, loss = 1.8819
epoch = 9, loss = 1.8834


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Определение модели
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(32 * 32 * 3, 6144)
        self.layer2 = nn.Linear(6144, 6144)
        self.layer3 = nn.Linear(6144, 100)
        self.layer4 = nn.Linear(100, 10)
        
        # Определяем активации
        self.relu = nn.ReLU()
        self.tanh = nn.Tanh()
        
    def forward(self, x):  # x - это тензор, а не DataLoader!
        x = x.view(x.size(0), -1)  # изменяем форму
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        x = self.tanh(self.layer3(x))
        x = self.layer4(x)  # последний слой без активации (CrossEntropyLoss включает softmax)
        return x

# Создание экземпляра модели
model = Model().to(device)

# Функция потерь и оптимизатор
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)  # 0.01 может быть слишком большим

# Обучение
for epoch in range(10):
    running_loss = 0.0
    for x_batch, y_batch in cifar10_loader_train:
        # Перемещаем данные на устройство
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)
        
        # Прямой проход (модель сама сделает view)
        pred = model(x_batch)
        
        # Вычисляем loss
        loss = loss_fn(pred, y_batch)
        
        # Обратный проход
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        
        running_loss += loss.item()
    
    avg_loss = running_loss / len(cifar10_loader_train)
    print(f"epoch = {epoch}, loss = {avg_loss:.4f}")

epoch = 0, loss = 2.0715
epoch = 1, loss = 1.9218
epoch = 2, loss = 1.8704
epoch = 3, loss = 1.8305
epoch = 4, loss = 1.8289
epoch = 5, loss = 1.8177
epoch = 6, loss = 1.8357
epoch = 7, loss = 1.8386
epoch = 8, loss = 1.8413
epoch = 9, loss = 1.8334


In [11]:

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

cifar10_test = datasets.CIFAR10(root='data', train=False, 
                                 transform=transform_test, download=True)
cifar10_loader_test = DataLoader(cifar10_test, batch_size=1024, shuffle=False)

# Модель
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 18, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(18)
        self.conv2 = nn.Conv2d(18, 72, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(72)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv3 = nn.Conv2d(72, 72, 3, padding=1)
        self.fc1 = nn.Linear(72 * 16 * 16, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, 10)
    
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool1(x)
        x = F.relu(self.conv3(x))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Функция для вычисления точности
def calculate_accuracy(model, dataloader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x_batch, y_batch in dataloader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            outputs = model(x_batch)
            _, predicted = torch.max(outputs.data, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()
    model.train()
    return 100 * correct / total

# Настройка TensorBoard
# Создаем директорию с временной меткой
log_dir = f"runs/cifar10_experiment_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
writer = SummaryWriter(log_dir)

# Инициализация модели, функции потерь и оптимизатора
loss_fn = nn.CrossEntropyLoss()
model = Model().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Замер времени обучения
start_time = time.time()
epoch_times = []


for epoch in range(20):
    epoch_start_time = time.time()
    running_loss = 0.0
    
    for batch_idx, (x_batch, y_batch) in enumerate(cifar10_loader_train):
        # Перемещаем данные на устройство
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)
        
        # Прямой проход
        pred = model(x_batch)
        
        # Вычисляем loss
        loss = loss_fn(pred, y_batch)
        
        # Обратный проход
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        
        running_loss += loss.item()
        
        # Логирование для каждого батча (каждые 50 батчей)
        if batch_idx % 50 == 0:
            global_step = epoch * len(cifar10_loader_train) + batch_idx
            writer.add_scalar('Loss/train_batch', loss.item(), global_step)
    
    # Вычисляем средний loss за эпоху
    avg_loss = running_loss / len(cifar10_loader_train)
    
    # Вычисляем точность на train и test
    train_accuracy = calculate_accuracy(model, cifar10_loader_train, device)
    test_accuracy = calculate_accuracy(model, cifar10_loader_test, device)
    
    # Замер времени эпохи
    epoch_time = time.time() - epoch_start_time
    epoch_times.append(epoch_time)
    
    # Логирование в TensorBoard
    writer.add_scalar('Loss/train_epoch', avg_loss, epoch)
    writer.add_scalar('Accuracy/train', train_accuracy, epoch)
    writer.add_scalar('Accuracy/test', test_accuracy, epoch)
    writer.add_scalar('Time/epoch', epoch_time, epoch)
    
    # Логирование параметров модели (гистограммы весов)
    for name, param in model.named_parameters():
        writer.add_histogram(f'Parameters/{name}', param, epoch)
        if param.grad is not None:
            writer.add_histogram(f'Gradients/{name}', param.grad, epoch)
    
    # Логирование learning rate
    current_lr = optimizer.param_groups[0]['lr']
    writer.add_scalar('Learning_rate', current_lr, epoch)
    
    # Вывод в консоль
    print(f"epoch {epoch:2d}: loss={avg_loss:.4f}, "
          f"train_acc={train_accuracy:.2f}%, test_acc={test_accuracy:.2f}%, "
          f"time={epoch_time:.2f}s")
    
    # Сохраняем лучшую модель
    if epoch == 0:
        best_test_acc = test_accuracy
        torch.save(model.state_dict(), 'best_model.pth')
    elif test_accuracy > best_test_acc:
        best_test_acc = test_accuracy
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"  -> Новая лучшая модель! Accuracy: {best_test_acc:.2f}%")

# Общее время обучения
total_time = time.time() - start_time

# Вывод статистики по времени
print(f"Общее время обучения: {total_time:.2f} секунд ({total_time/60:.2f} минут)")
print(f"Среднее время на эпоху: {sum(epoch_times)/len(epoch_times):.2f} секунд")
print(f"Минимальное время эпохи: {min(epoch_times):.2f} секунд")
print(f"Максимальное время эпохи: {max(epoch_times):.2f} секунд")
print(f"Лучшая точность на тесте: {best_test_acc:.2f}%")

# Закрываем TensorBoard writer
writer.close()

# Сохраняем метрики в файл
with open('training_metrics.txt', 'w') as f:
    f.write(f"Total training time: {total_time:.2f} seconds\n")
    f.write(f"Average epoch time: {sum(epoch_times)/len(epoch_times):.2f} seconds\n")
    f.write(f"Best test accuracy: {best_test_acc:.2f}%\n")

Files already downloaded and verified
epoch  0: loss=1.7887, train_acc=39.74%, test_acc=42.85%, time=21.72s
epoch  1: loss=1.4070, train_acc=52.83%, test_acc=55.04%, time=21.41s
  -> Новая лучшая модель! Accuracy: 55.04%
epoch  2: loss=1.2432, train_acc=57.53%, test_acc=59.72%, time=21.27s
  -> Новая лучшая модель! Accuracy: 59.72%
epoch  3: loss=1.1211, train_acc=61.95%, test_acc=63.66%, time=21.76s
  -> Новая лучшая модель! Accuracy: 63.66%
epoch  4: loss=1.0285, train_acc=62.34%, test_acc=65.28%, time=21.40s
  -> Новая лучшая модель! Accuracy: 65.28%
epoch  5: loss=0.9759, train_acc=66.95%, test_acc=69.60%, time=23.20s
  -> Новая лучшая модель! Accuracy: 69.60%
epoch  6: loss=0.9254, train_acc=68.85%, test_acc=70.57%, time=23.90s
  -> Новая лучшая модель! Accuracy: 70.57%
epoch  7: loss=0.8735, train_acc=70.59%, test_acc=72.01%, time=22.44s
  -> Новая лучшая модель! Accuracy: 72.01%
epoch  8: loss=0.8367, train_acc=71.44%, test_acc=72.65%, time=21.15s
  -> Новая лучшая модель! Accur

In [20]:
# Определите device в самом начале
device = torch.device('cpu')  # Явно используем CPU
print(f"Используется устройство: {device}")

# Модель
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 18, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(18)
        self.conv2 = nn.Conv2d(18, 72, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(72)
        self.pool1 = nn.AvgPool2d(2, 2)
        self.conv3 = nn.Conv2d(72, 72, 3, padding=1)
        self.fc1 = nn.Linear(72 * 16 * 16, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, 10)
    
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool1(x)
        x = F.relu(self.conv3(x))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Функция для вычисления точности
def calculate_accuracy(model, dataloader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x_batch, y_batch in dataloader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            outputs = model(x_batch)
            _, predicted = torch.max(outputs.data, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()
    model.train()
    return 100 * correct / total

# Настройка TensorBoard
log_dir = f"runs/cifar10_experiment_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
writer = SummaryWriter(log_dir)

# Инициализация модели, функции потерь и оптимизатора
loss_fn = nn.CrossEntropyLoss()
model = Model().to(device)  # Изменено: используем device вместо "cpu"
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Замер времени обучения
start_time = time.time()
epoch_times = []

for epoch in range(20):
    epoch_start_time = time.time()
    running_loss = 0.0
    
    for batch_idx, (x_batch, y_batch) in enumerate(cifar10_loader_train):
        # Перемещаем данные на устройство
        x_batch = x_batch.to(device)  # Изменено: используем device
        y_batch = y_batch.to(device)  # Изменено: используем device
        
        # Прямой проход
        pred = model(x_batch)
        
        # Вычисляем loss
        loss = loss_fn(pred, y_batch)
        
        # Обратный проход
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        
        running_loss += loss.item()
        
        # Логирование для каждого батча (каждые 50 батчей)
        if batch_idx % 50 == 0:
            global_step = epoch * len(cifar10_loader_train) + batch_idx
            writer.add_scalar('Loss/train_batch', loss.item(), global_step)
    
    # Вычисляем средний loss за эпоху
    avg_loss = running_loss / len(cifar10_loader_train)
    
    # Вычисляем точность на train и test
    train_accuracy = calculate_accuracy(model, cifar10_loader_train, device)
    test_accuracy = calculate_accuracy(model, cifar10_loader_test, device)
    
    # Замер времени эпохи
    epoch_time = time.time() - epoch_start_time
    epoch_times.append(epoch_time)
    
    # Логирование в TensorBoard
    writer.add_scalar('Loss/train_epoch', avg_loss, epoch)
    writer.add_scalar('Accuracy/train', train_accuracy, epoch)
    writer.add_scalar('Accuracy/test', test_accuracy, epoch)
    writer.add_scalar('Time/epoch', epoch_time, epoch)
    
    # Логирование параметров модели (гистограммы весов)
    for name, param in model.named_parameters():
        writer.add_histogram(f'Parameters/{name}', param, epoch)
        if param.grad is not None:
            writer.add_histogram(f'Gradients/{name}', param.grad, epoch)
    
    # Логирование learning rate
    current_lr = optimizer.param_groups[0]['lr']
    writer.add_scalar('Learning_rate', current_lr, epoch)
    
    # Вывод в консоль
    print(f"epoch {epoch:2d}: loss={avg_loss:.4f}, "
          f"train_acc={train_accuracy:.2f}%, test_acc={test_accuracy:.2f}%, "
          f"time={epoch_time:.2f}s")
    
    # Сохраняем лучшую модель
    if epoch == 0:
        best_test_acc = test_accuracy
        torch.save(model.state_dict(), 'best_model.pth')
    elif test_accuracy > best_test_acc:
        best_test_acc = test_accuracy
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"  -> Новая лучшая модель! Accuracy: {best_test_acc:.2f}%")

# Общее время обучения
total_time = time.time() - start_time

# Вывод статистики по времени
print(f"Общее время обучения: {total_time:.2f} секунд ({total_time/60:.2f} минут)")
print(f"Среднее время на эпоху: {sum(epoch_times)/len(epoch_times):.2f} секунд")
print(f"Минимальное время эпохи: {min(epoch_times):.2f} секунд")
print(f"Максимальное время эпохи: {max(epoch_times):.2f} секунд")
print(f"Лучшая точность на тесте: {best_test_acc:.2f}%")

# Закрываем TensorBoard writer
writer.close()

# Сохраняем метрики в файл
with open('training_metrics.txt', 'w') as f:
    f.write(f"Total training time: {total_time:.2f} seconds\n")
    f.write(f"Average epoch time: {sum(epoch_times)/len(epoch_times):.2f} seconds\n")
    f.write(f"Best test accuracy: {best_test_acc:.2f}%\n")

Используется устройство: cpu
epoch  0: loss=1.7996, train_acc=41.66%, test_acc=43.94%, time=83.55s
epoch  1: loss=1.4318, train_acc=50.74%, test_acc=53.39%, time=84.24s
  -> Новая лучшая модель! Accuracy: 53.39%
epoch  2: loss=1.2911, train_acc=55.88%, test_acc=59.27%, time=86.49s
  -> Новая лучшая модель! Accuracy: 59.27%
epoch  3: loss=1.1605, train_acc=59.81%, test_acc=61.88%, time=84.59s
  -> Новая лучшая модель! Accuracy: 61.88%
epoch  4: loss=1.0874, train_acc=63.12%, test_acc=65.67%, time=83.10s
  -> Новая лучшая модель! Accuracy: 65.67%
epoch  5: loss=1.0121, train_acc=64.83%, test_acc=67.05%, time=80.55s
  -> Новая лучшая модель! Accuracy: 67.05%
epoch  6: loss=0.9607, train_acc=66.05%, test_acc=68.47%, time=80.13s
  -> Новая лучшая модель! Accuracy: 68.47%
epoch  7: loss=0.9099, train_acc=68.65%, test_acc=71.25%, time=80.22s
  -> Новая лучшая модель! Accuracy: 71.25%
epoch  8: loss=0.8700, train_acc=68.70%, test_acc=70.31%, time=80.18s
epoch  9: loss=0.8436, train_acc=71.23%,